# Load a pre-trained model

## A. Import an untrained model

In [ ]:
import jax
import jax.numpy as jnp
from jax import Array
from jaxtyping import PyTree

import flax.nnx as nnx

from src.config import ModelConfig
from src.model.model import NanoLLM

In [ ]:
# Load the configuration that will be used to build a new model
model_config = ModelConfig()
print("Model config:")
print(f"  maxlen: {model_config.maxlen}") # inference constraint
print(f"  num_transformer_blocks: {model_config.num_transformer_blocks}") # model depth
print(f"  feed_forward_dim: {model_config.feed_forward_dim}") # model capacity
print(f"  vocab_size: {model_config.vocab_size}") # validation check
print(f"  embed_dim: {model_config.embed_dim}")
print(f"  num_heads: {model_config.num_heads}")

In [ ]:
# Build a new model from scratch (untrained) using the imported configuration
nano_model = NanoLLM(model_config)

# inspect the model
# nano_model

## B. Capture characteristics of the untrained model (BEFORE)

In [ ]:
# STATE - all values (potentially millions of numbers)

# It's very important that we don't use a reference to a structure that will be
# mutated by nxx.update(), such as the States that are returned from nnx.state() and nnx.split().
def state_snapshot(the_model: NanoLLM) -> PyTree[Array]:

    mutable_state = nnx.state(the_model)

    # tree_map forces an independent and deep copy of the values, creating a new Param object using pytree unflatten.
    snapshot = jax.tree_util.tree_map(lambda x: jnp.array(x), mutable_state)
    return snapshot

initial_state = state_snapshot(nano_model)

# inspect the state
# initial_state

In [ ]:
# NORMS - measure of magnitude (consumes less memory)

# Create an independent transformation of the deep copy state
def norms_snapshot(model_state: PyTree[Array]) -> PyTree[Array]:
    
    norms = jax.tree_util.tree_map(jnp.linalg.norm, model_state)
    return norms

initial_norms = norms_snapshot(initial_state)

# inspect the norms
# initial_norms

### Notes on Norms
- Each layer's norm is the overall size (level of magnitude) of that layer's weights.
A layer contains hundreds or thousands of individual numbers (weights). The norm collapses all of those into a single number that answers the question: How big are these weights overall? It's like asking "how loud is this song?" instead of describing every instrument. It doesn't tell you the shape or direction of the weights - only their overall magnitude.It's a way to compare two models based on this measure of magnitude.
- Norms of two models are compared using a ratio (e.g., after_training:before_training). This gives a single number per layer that describes how much the scale changed. 
    - norm ratio = 1.0 → the layer is the same size after the udpate as it was before the update (no change)
    - norm ratio = 2.0 → the layer's weights after update are twice the size they were before the update
    - norm ratio = 0.5 → the layer's weights after update are half the size they were before the update
- !! The norm ratio being close to 1.0 doesn't mean the weights are similar. It only means the magnitudes are similar. If a model is well-constructed, it will be designed to initialize weights at the same scale as a well-trained model. So it's not unexpected that there is very little differnence in the norms of a well-constructed, untrained model and a model that has been trained.

### Inappropriate ways of capturing state
These are INCORRECT because they return references to objects that will be mutated by nxx.update().

  
- **Storing nnx.state directly**:
  ```python
  initial_state_snapshot = nnx.state(nano_model)

  ```

  
  
- **nnx.split()**:
  ```python
  initial_graphdef, initial_state_snapshot = nnx.split(nano_model)

  # graphdef:
  #    the “blueprint” of a model. It describes the structure and types of layers,
  #    but does not contain actual numbers or weights.

  # state snapshot:
  #    reference to the collection of all the model’s numerical values that
  #    can change (e.g. weights, biases, batch stats, optimizer buffers).
  
  ```



## C. Use checkpoint weights to update the model

- This requires a knowledge of the shape of the model, which is why the model was instantiated before this section.
- This notebook updates the model by manually walking through multiple steps. A consolidated straightforward approach is now available to callers. See src/checkpoint.py

In [ ]:
import orbax
from orbax import checkpoint

# to load a model on one machine (my CPU) when the model was originally trained on many machines (likely GPUs)
from jax.sharding import SingleDeviceSharding

from src.checkpoint import get_latest_checkpoint
from src.paths import CHECKPOINTS_DIR

In [ ]:
# 1. Locate and define current machine on which you will run the model
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

In [ ]:
# 2. Add capability to retrieve the checkpoint onto as single CPU
restore_args = jax.tree_util.tree_map(

    # for each array of parameters, restore that array to the single CPU device 
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),

    # returns the shape of the model, not the actual model
    nnx.state(nano_model)
)

In [ ]:
# 3. Load the most recent checkpoint
checkpoint_path = get_latest_checkpoint()
if checkpoint_path is None:
    raise FileNotFoundError(f"No checkpoint bundles found in {CHECKPOINTS_DIR}")

checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [ ]:
# 4. Update the model with the loaded checkpoint

restored_state = checkpointer.restore(
    checkpoint_path / "weights.orbax", # the weights are stored in an orbax subdirectory
    item=nnx.state(nano_model),
    restore_args=restore_args)

# DESTRUCTIVE!! This mutates state so make sure you made independent copies of the initial state before executing this.
nnx.update(nano_model, restored_state)

## D. Capture characteristics of the updated, trained model (AFTER)

In [ ]:
# STATE - all values (potentially millions of numbers)
resulting_state = state_snapshot(nano_model)

# alternative (take care because this references a state that can be mutated by other operations)
# resulting_state = nnx.state(model)

In [ ]:
# NORMS - measure of magnitude (consumes less memory)
resulting_norms = norms_snapshot(resulting_state)

## E. Compare BEFORE and AFTER

### Comparison of Norms (magnitude of weights in each layer)

In [ ]:
# Compare per-layer norms before and after checkpoint loading
# Arguments are of the type PyTree[Array], a frozen snapshot of array values (not a live mutable state)
def compare_norms(norms_before: PyTree[Array], norms_after: PyTree[Array]) -> None:
    norm_ratios = jax.tree_util.tree_map(
        lambda a, b: b / a,
        norms_before, norms_after
    )

    ratio_list = jax.tree_util.tree_leaves(norm_ratios)
    total_layers = len(ratio_list)
    
    ratio_leaves = jnp.stack(ratio_list)
    finite_mask = jnp.isfinite(ratio_leaves)
    finite_ratios = ratio_leaves[finite_mask]
    infinite_layers = int((~finite_mask).sum())
    compared_layers = int(finite_mask.sum())

    median = float(jnp.median(finite_ratios))
    min_ratio = float(jnp.min(finite_ratios))
    max_ratio = float(jnp.max(finite_ratios))

    # Improvement: add data structure that stores all values so the printout logic can be stored in separate method
    print("Weight magnitude (aka norm) comparison:\n")
    print("  Remember this is a comparison of magnitude (not quality) and untrained models are designed to initialize with magnitudes similar to trained models.")
    print("\n----------\n")
    
    print(f"  Median ratio:  {median:.4f}")
    print(f"    This is the most representative ratio across all layers (robust to outliers).")
    print(f"    Range: 0.0 to infinity")
    print(f"      ~1.0  = weights are similar in scale")
    print(f"      << 1.0 = weights shrank substantially overall")
    print(f"      >> 1.0 = weights grew substantially overall")
    print()

    print(f"  Min ratio:     {min_ratio:.4f}")
    print(f"    This is the ratio for the layer that shrank the most (or grew the least).")
    print(f"    Range: 0.0 to infinity")
    print(f"      ~0.0  = at least one layer nearly collapsed to zero (possible dead layer)")
    print(f"      ~1.0  = even the most-shrunken layer barely changed in scale")
    print(f"      > 1.0 = every layer grew; none shrank")
    print()

    print(f"  Max ratio:     {max_ratio:.4f}")
    print(f"    This is the ratio for the layer that grew the most (or shrank the least).")
    print(f"    Range: 0.0 to infinity")
    print(f"      < 1.0  = every layer shrank; none grew")
    print(f"      ~1.0   = even the most-grown layer barely changed in scale")
    print(f"      >> 1.0 = at least one layer's weights grew dramatically")
    print("\n----------\n")

    print(f"  {infinite_layers} layers (of {total_layers} total layers) were skipped due to presence of zero in the denominator.")
    print(f"  Total layers:   {total_layers}")
    print(f"  Compared:       {compared_layers}")
    print(f"  Skipped:        {infinite_layers}")
    print("\n----------\n")

    percentage = abs(median - 1.0) * 100
    if percentage < 1.0:
        summary = f"The models are very similar. Median weight size differs by only {percentage:.2f}%."
    elif percentage < 5.0:
        summary = f"The models are moderately different. Median weight size differs by {percentage:.1f}%."
    else:
        direction = "grown" if median > 1.0 else "shrunk"
        summary = f"The models are noticeably different. Median weight size has {direction} by {percentage:.1f}%."

    print(f"SUMMARY: {summary}")
    print(f"         NB: Models that have similar norms does not mean that they have identical weights. The states of the model before and after should be analyzed to examine parameters more closely.")

In [ ]:
# norm ratios are reported as after_training : before_training
compare_norms(initial_norms, resulting_norms)

### Comparison of States (actual parameters)

In [ ]:
def analyze_state_changes(state1: PyTree[Array], state2: PyTree[Array]) -> None:
    """Analyze differences between two model states"""

    # Calculate differences for each parameter array
    diffs = jax.tree_util.tree_map(
        lambda x, y: jnp.abs(x - y),
        state1, state2
    )

    # Flatten to get all individual statistics
    all_diffs = jax.tree_util.tree_leaves(diffs)
    total_params = sum(x.size for x in all_diffs)
    changed_params = int(sum((x > 1e-8).sum() for x in all_diffs))  # threshold for "changed"
    unchanged_params = total_params - changed_params
    percent_changed = 100 * changed_params / total_params

    # This materializes the full diff array (potentially hundreds of millions of floats) for larger models.
    #   e.g., A model with 100M parameters would consume roughly 400MB for this comparison.
    #   Potential improvement: Compute statistics per-leaf and aggregate (e.g., mean-of-means) instead of concatenating all figures.
    all_diff_values = jnp.concatenate([x.flatten() for x in all_diffs])

    mean_change = float(jnp.mean(all_diff_values))
    median_change = float(jnp.median(all_diff_values))
    max_change = float(jnp.max(all_diff_values))
    min_change = float(jnp.min(all_diff_values))

    # Improvement: add data structure that stores all values so the printout logic can be stored in separate method
    print("Model parameter comparison:\n")
    print("  Remember this compares actual weight values, not just magnitudes.")
    print("  Values near zero indicate the model is unchanged in that respect.")
    print("\n----------\n")

    print(f"  Percentage changed: {percent_changed:.2f}%")
    print(f"    Fraction of individual weight values that shifted by more than 1e-8.")
    print(f"    Range: 0% to 100%")
    print(f"      ~0%   = models are nearly identical (weights barely changed)")
    print(f"      ~50%  = roughly half of all weights were updated")
    print(f"      ~100% = almost every weight was updated")
    print()

    print(f"  Total parameters:   {total_params:,}")
    print(f"  Changed:            {changed_params:,}")
    print(f"  Unchanged:          {unchanged_params:,}")
    print("\n----------\n")

    print(f"  Median absolute change: {median_change:.6f}")
    print(f"    The most representative per-weight change (robust to outliers).")
    print(f"    Range: 0.0 to infinity")
    print(f"      ~0.0     = the typical weight barely moved")
    print(f"      small    = fine-grained updates; weights shifted only slightly on average")
    print(f"      large    = weights shifted substantially on average")
    print()

    print(f"  Mean absolute change:   {mean_change:.6f}")
    print(f"    Average per-weight change across all parameters.")
    print(f"    Range: 0.0 to infinity")
    print(f"      ~0.0            = weights are nearly identical overall")
    print(f"      >> median       = a few large outlier updates are skewing the average")
    print(f"      ~ median        = changes are uniformly distributed across weights")
    print()

    print(f"  Max absolute change:    {max_change:.6f}")
    print(f"    The single largest weight shift anywhere in the model.")
    print(f"    Range: 0.0 to infinity")
    print(f"      ~0.0  = no weight changed significantly")
    print(f"      large = at least one weight shifted dramatically (may indicate an outlier layer)")
    print()

    print(f"  Min absolute change:    {min_change:.6f}")
    print(f"    The smallest weight shift (includes unchanged weights at ~0.0).")
    print(f"    Range: 0.0 to infinity")
    print(f"      ~0.0  = at least one weight is essentially unchanged (expected if any params frozen)")
    print(f"      > 0.0 = every single weight shifted by at least this amount")
    print("\n----------\n")

    if percent_changed < 1.0:
        summary = f"Almost no weights changed ({percent_changed:.2f}%). The two model states are nearly identical."
    elif percent_changed < 50.0:
        summary = f"A minority of weights changed ({percent_changed:.1f}%). Partial update or sparse fine-tuning likely."
    elif median_change < 1e-4:
        summary = f"{percent_changed:.1f}% of weights changed, but by very small amounts (median: {median_change:.2e}). The models are subtly different."
    else:
        summary = f"{percent_changed:.1f}% of weights changed, with a median shift of {median_change:.6f}. The models are substantially different."

    print(f"SUMMARY: {summary}")
    print(f"         NB: A high percentage of changed weights is expected when loading a trained checkpoint into an untrained model.")

In [ ]:
analyze_state_changes(initial_state, resulting_state)

### Compare using imported python modules

In [ ]:
# Consolidated API was developed based on the work in the above sections.
from src.compare import (
    state_snapshot,
    compare_norms,
    compare_states,
    format_norms_comparison,
    format_state_comparison,
)

In [ ]:
# initial_state and resulting_state are already saved to deep copies
# so we don't need to worry about state variables getting mutated by subsequent training
norms_report = compare_norms(initial_state, resulting_state)
print(format_norms_comparison(norms_report))

In [ ]:
states_report = compare_states(initial_state, resulting_state)
print(format_state_comparison(states_report))